# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima12aa/fa-ml/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
import pandas as pd
df = pd.read_csv("content_refresh_anonymized.csv")
df.columns.tolist()
#one row = one page. There are no explicit dates/time frames provided in the csv file. Instead the data available is relative (since 90 days or 30 days) as is seen by these columns



['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [ ]:
print((df["impressions_last_30d"] == df["impressions_prev_30d"]).mean())
#however, it is important to note the ovelap between the entries in impressions_last_30d and impressions_prev_30d is only around 5.3 percent. this shows that both these columns are not duplicates and hence the data available was recorded at different times. These are genuinely separate windows, not just relabeled duplicates.

0.05313333333333333


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
#df["computed_pct_check"] = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"] * 100
#df[["trend_pct", "computed_pct_check"]].head(10)
#df["provider_used"].unique()
#df["model_used"].unique()
df["provider_used"].isna().mean()
df["ai_traffic_pct"].isna().mean()
(df["ai_traffic_pct"] == 0).mean()

np.float64(0.9356666666666666)

Feature — real, observed signals knowable before the outcome: content_age_days, days_since_last_update, search_volume, competition, cpc, word_count, char_count, impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions, impressions_last_30d/prev_30d, clicks_last_30d/prev_30d, sessions_last_30d/prev_30d, ctr, avg_position, engagement_rate, scroll_rate.

Label — trend_direction (the source of is_declining_label, our proxy target).

Context — content_id, client_id (pseudonymized identifiers, no predictive pattern); age_tier, freshness_tier, word_count_tier, char_count_tier, impression_tier, position_tier (bucketed duplicates of raw numeric columns already used as features — redundant, kept only for human-readable reason codes).

Excluded —

trend_pct: verified numerically (via (last_30d - prev_30d)/prev_30d * 100) to be the exact value trend_direction is derived from. Using it would be leakage — the model would learn to copy the label instead of finding real signal.
provider_used, model_used: 71.5% missing, and even where present, risk acting as a confound — any correlation between AI model and decline may really reflect when that model was adopted, not genuine content quality.
ai_traffic_pct: no missing values, but 93.56% of rows are exactly 0, meaning very little real signal for the vast majority of pages — consistent with the lane guide's warning that AI-session data is generally sparse. Excluded rather than treated as a normal feature.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# Claim 1: one row = one page (grain check)
print("Shape:", df.shape)
print("Unique content_id count:", df["content_id"].nunique())
# If nunique == number of rows, confirms one row per page, no duplicates

# Claim 2: last_30d and prev_30d are genuinely separate windows, not duplicates
overlap_rate = (df["impressions_last_30d"] == df["impressions_prev_30d"]).mean()
print("Fraction identical between last_30d and prev_30d:", overlap_rate)
# ~5.3% identical -> genuinely separate windows, not a duplicated column

# Claim 3: trend_pct is derived from last_30d/prev_30d, and trend_direction from trend_pct -> leakage
df["computed_pct_check"] = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"] * 100
print(df[["trend_pct", "computed_pct_check"]].head(10))
# Near-identical values confirm trend_pct's formula, and therefore why it must be excluded

# Claim 4: provider_used / model_used are heavily missing
print("provider_used missing rate:", df["provider_used"].isna().mean())
print("model_used unique values:", df["model_used"].unique())

# Claim 5: ai_traffic_pct has no NaNs but is mostly zero
print("ai_traffic_pct missing rate:", df["ai_traffic_pct"].isna().mean())
print("ai_traffic_pct == 0 rate:", (df["ai_traffic_pct"] == 0).mean())

# Claim 6: avg_position uses 0 as a sentinel for "no data," distorting the mean if left in
print("Rows with avg_position == 0:", (df["avg_position"] == 0).sum())
print("Mean avg_position including zeros:", df["avg_position"].mean())
print("Mean avg_position excluding zeros:", df[df["avg_position"] > 0]["avg_position"].mean())

Shape: (30000, 45)
Unique content_id count: 30000
Fraction identical between last_30d and prev_30d: 0.05313333333333333
   trend_pct  computed_pct_check
0      -41.4          -41.438703
1      -57.7          -57.717667
2      -60.9          -60.880276
3      -13.8          -13.789824
4      -34.7          -34.733416
5      -38.9          -38.850347
6      -92.3          -92.307692
7        0.6            0.632911
8      -58.8          -58.808215
9      -29.2          -29.213483
provider_used missing rate: 0.7146
model_used unique values: ['gemini-2.5-flash' 'gemini-3-flash-preview' nan 'gpt-4o-mini' 'unknown'
 'gpt-5-mini']
ai_traffic_pct missing rate: 0.0
ai_traffic_pct == 0 rate: 0.9356666666666666
Rows with avg_position == 0: 1205
Mean avg_position including zeros: 16.342380000000002
Mean avg_position excluding zeros: 17.026268449383576


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.